<a href="https://colab.research.google.com/github/true-cpu/DoAnKPDL/blob/main/DOANKPDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score

# Giải nén tệp tin
zip_path = 'archive.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data')
    print("Giải nén thành công.")
else:
    print("Vui lòng đảm bảo tệp archive.zip đã được tải lên.")

In [ ]:
import pandas as pd
import os

# Tải dữ liệu từ thư mục 'data'
file_path = 'data/bot_detection_data.csv'
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Tải dữ liệu thành công.")
    display(df.head())
else:
    print(f"Không tìm thấy tệp tin tại: {file_path}")

### 1. Tiền xử lý dữ liệu
Chúng ta sẽ chọn các đặc trưng hành vi quan trọng: `Retweet Count`, `Mention Count`, `Follower Count` và chuẩn hóa chúng.

In [ ]:
# Xử lý giá trị thiếu
df_clean = df.dropna().copy()

# Chọn đặc trưng hành vi
features = ['Retweet Count', 'Mention Count', 'Follower Count']
X = df_clean[features]

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dữ liệu đã được chuẩn hóa. Kích thước:", X_scaled.shape)

### 2. Xác định số lượng cụm tối ưu (Phương pháp Elbow)

In [ ]:
wcss = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, wcss, 'bx-')
plt.xlabel('Số lượng cụm (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Phương pháp Elbow để tìm k tối ưu')
plt.show()

### 3. Thực hiện gom cụm và phân tích hành vi bất thường
Giả sử k=4 dựa trên cấu trúc dữ liệu để phân loại chi tiết hơn.

In [ ]:
# Áp dụng K-Means với k=4
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
df_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# Phân tích đặc trưng trung bình của mỗi cụm
cluster_analysis = df_clean.groupby('Cluster')[features].mean()
print("Đặc trưng trung bình của từng cụm:")
display(cluster_analysis)

# Trực quan hóa kết quả bằng biểu đồ 3D
import plotly.express as px
fig = px.scatter_3d(df_clean.sample(2000), x='Retweet Count', y='Mention Count', z='Follower Count',
                    color='Cluster', title='Phân cụm hành vi người dùng (Mẫu 2000 dòng)')
fig.show()

In [ ]:
def name_clusters(cluster_id):
    # Logic đặt tên dựa trên cluster_analysis đã quan sát
    mapping = {
        0: "Lan tỏa & Uy tín (Influencers)",
        1: "Spam/Bot tiềm năng (High Mentions)",
        2: "Uy tín nhưng thụ động (High Followers)",
        3: "Người dùng phổ thông (Low Activity)"
    }
    return mapping.get(cluster_id, f"Cụm {cluster_id}")

# Áp dụng đặt tên
df_clean['Cluster Name'] = df_clean['Cluster'].apply(name_clusters)

# Hiển thị thống kê theo tên cụm
summary = df_clean.groupby('Cluster Name')[features].mean().sort_values(by='Retweet Count', ascending=False)
display(summary)

In [ ]:
# Trực quan hóa lại với tên cụm có ý nghĩa
fig_named = px.scatter_3d(
    df_clean.sample(min(2000, len(df_clean))),
    x='Retweet Count', y='Mention Count', z='Follower Count',
    color='Cluster Name',
    title="Phân loại hành vi người dùng theo nhóm ý nghĩa",
    labels={'Cluster Name': 'Nhóm người dùng'}
)
fig_named.show()

### 4. So sánh với nhãn Bot Label thực tế
Chúng ta sẽ kiểm tra xem các cụm đã phân loại có thực sự chứa nhiều Bot (nhãn 1) hay không.

In [ ]:
# Lập bảng chéo giữa Tên cụm và Bot Label
bot_comparison = pd.crosstab(df_clean['Cluster Name'], df_clean['Bot Label'], normalize='index') * 100

# Trực quan hóa
plt.figure(figsize=(12, 6))
bot_comparison.plot(kind='bar', stacked=True, color=['#66b3ff', '#ff9999'], figsize=(12,6))
plt.title('Tỷ lệ Bot thực tế trong mỗi cụm hành vi')
plt.ylabel('Phần trăm (%)')
plt.xlabel('Nhóm hành vi')
plt.legend(title='Thực tế', labels=['Người thật (0)', 'Bot (1)'])
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

display(bot_comparison)

### 5. Đánh giá độ tách biệt cụm (Silhouette Score)
Chỉ số Silhouette giúp định lượng chất lượng của việc phân cụm.

In [ ]:
from sklearn.metrics import silhouette_score

# Tính toán Silhouette Score trên một mẫu dữ liệu để tối ưu tốc độ
sample_size = min(10000, len(X_scaled))
X_sample = X_scaled[np.random.choice(X_scaled.shape[0], sample_size, replace=False)]
labels_sample = kmeans.predict(X_sample)

score = silhouette_score(X_sample, labels_sample)
print(f"Silhouette Score (mẫu {sample_size} dòng): {score:.4f}")

if score > 0.5:
    print("Nhận xét: Các cụm có độ tách biệt tốt.")
elif score > 0.2:
    print("Nhận xét: Các cụm có sự phân hóa nhưng vẫn còn chồng lấn.")
else:
    print("Nhận xét: Độ tách biệt giữa các cụm thấp.")

### 6. Thử nghiệm thay đổi số lượng cụm để tối ưu Silhouette Score
Chúng ta sẽ lặp qua các giá trị k khác nhau và đánh giá chất lượng phân cụm.

In [ ]:
results = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)

    # Lấy mẫu để tính toán nhanh
    indices = np.random.choice(X_scaled.shape[0], 5000, replace=False)
    sh_score = silhouette_score(X_scaled[indices], labels[indices])
    results.append({'k': k, 'silhouette': sh_score})
    print(f"k={k} | Silhouette Score: {sh_score:.4f}")

# Vẽ biểu đồ so sánh
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['silhouette'] for r in results], marker='o')
plt.title('Silhouette Score theo số lượng cụm k')
plt.xlabel('Số lượng cụm (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

### 7. Cập nhật mô hình với k=6 và đặt tên cụm mới
Chúng ta sẽ áp dụng k=6 để có độ phân giải dữ liệu tốt hơn.

In [ ]:
# Áp dụng K-Means với k=6
kmeans_v2 = KMeans(n_clusters=6, init='k-means++', random_state=42, n_init=10)
df_clean['Cluster_v2'] = kmeans_v2.fit_predict(X_scaled)

# Phân tích đặc trưng để đặt tên
analysis_v2 = df_clean.groupby('Cluster_v2')[features].mean().sort_values(by='Retweet Count', ascending=False)
display(analysis_v2)

def name_clusters_v2(cluster_id):
    # Tên gợi ý dựa trên đặc trưng hành vi của 6 cụm
    mapping = {
        0: "Lan tỏa cực cao (Super Influencers)",
        1: "Tương tác spam (High Mentions)",
        2: "Uy tín lớn nhưng ít hoạt động",
        3: "Người dùng phổ thông năng động",
        4: "Tài khoản mới/Thụ động",
        5: "Nhóm trung gian (Moderate Activity)"
    }
    return mapping.get(cluster_id, f"Cụm {cluster_id}")

df_clean['Cluster Name v2'] = df_clean['Cluster_v2'].apply(name_clusters_v2)

# Trực quan hóa 3D cho 6 cụm mới
fig_v2 = px.scatter_3d(
    df_clean.sample(min(2000, len(df_clean))),
    x='Retweet Count', y='Mention Count', z='Follower Count',
    color='Cluster Name v2',
    title="Phân loại chi tiết hành vi với k=6"
)
fig_v2.show()

### 8. Phân bổ số lượng người dùng theo từng cụm (k=6)
Chúng ta sẽ đếm số lượng tài khoản thuộc mỗi nhóm hành vi đã phân loại.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Đếm số lượng trong mỗi cụm
cluster_counts = df_clean['Cluster Name v2'].value_counts().reset_index()
cluster_counts.columns = ['Cluster Name', 'Count']

# Vẽ biểu đồ
plt.figure(figsize=(12, 6))
sns.barplot(data=cluster_counts, x='Count', y='Cluster Name', palette='viridis')

# Thêm nhãn số lượng cụ thể lên cột
for i, count in enumerate(cluster_counts['Count']):
    plt.text(count + 50, i, str(count), va='center')

plt.title('Số lượng người dùng trong mỗi cụm hành vi (k=6)')
plt.xlabel('Số lượng tài khoản')
plt.ylabel('Tên cụm')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# Hiển thị tỷ lệ phần trăm
cluster_counts['Percentage (%)'] = (cluster_counts['Count'] / len(df_clean) * 100).round(2)
display(cluster_counts)

### 9. So sánh tỷ lệ Bot Label thực tế trên 6 cụm mới
Chúng ta kiểm tra mức độ hiệu quả của việc tăng số cụm trong việc tách biệt tài khoản Bot.

In [ ]:
# Lập bảng chéo cho k=6
bot_comparison_v2 = pd.crosstab(df_clean['Cluster Name v2'], df_clean['Bot Label'], normalize='index') * 100

# Trực quan hóa
plt.figure(figsize=(12, 6))
bot_comparison_v2.plot(kind='bar', stacked=True, color=['#66b3ff', '#ff9999'], figsize=(12, 6))
plt.title('Tỷ lệ Bot thực tế trong 6 cụm hành vi mới (k=6)')
plt.ylabel('Phần trăm (%)')
plt.xlabel('Nhóm hành vi')
plt.legend(title='Thực tế', labels=['Người thật (0)', 'Bot (1)'])
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

display(bot_comparison_v2)